# Wolf-of-Wallstreet — QuantileTCN / DMN overlay training (Colab)

Trains the **sizing overlay** for the 4h TS-momentum sleeve, both objectives, then runs the
book-level OOS A/B automatically:

1. **`pinball`** — calibrated quantile forecast (edge = p50, uncertainty = p90−p10)
2. **`sharpe`** — DMN position head trained directly on NET Sharpe with a turnover cost term
   (Lim/Zohren/Roberts 2019)

The overlay only **scales** the sleeve's positions (floor 0.25 … 1.0) — it never originates and
never flips, so an uninformative model degrades to ≈ the unmodified sleeve.

**Promotion bar (decided up-front, no peeking):** keep an objective only if its OOS test-tail
book Sharpe beats the no-overlay baseline by **≥ +0.05**. Otherwise park it — same verdict the
meta-labeling experiment got.

**Before running:** upload the CURRENT repo to Drive (`MyDrive/trading-agent`). Files that MUST
be fresh (all changed 2026-07-02): `scripts/train_quantile_tcn.py`, `scripts/pretrain.py`,
`backend/models/sequence.py`, `backend/features/pipeline.py`, `backend/agents/improved_model.py`,
`backend/signals/feature_spec.py`, `backend/backtest/engine.py`, `backend/strategies/ts_momentum.py`.
Cell 1 verifies this. No `.env` needed — this run is crypto-only (no broker keys).

**Runtime:** pick a GPU runtime (T4 is fine). Rough timings: data prefetch ~5–10 min first time
(then cached on Drive), feature build ~2–4 min/symbol first time (then cached), pinball run
~1.5–2.5 h for 3 seeds × 10 epochs, sharpe run ~20–40 min (48-bar stride = far fewer windows).

### 1) Mount Drive + verify the uploaded files are current

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/trading-agent"   # <-- edit if your folder differs

from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR); sys.path.insert(0, os.path.join(PROJECT_DIR, 'backend'))
sys.path.insert(0, os.path.join(PROJECT_DIR, 'scripts'))
print('Working dir:', os.getcwd())

# Freshness check — every marker below is from the 2026-07-02 session. Re-upload whatever fails.
def has(path, *markers):
    txt = open(path, encoding='utf-8').read()
    return [m for m in markers if m not in txt]
miss = {}
miss['train_quantile_tcn.py'] = has('scripts/train_quantile_tcn.py',
    'DMNPositionNet', '--objective', 'gated_positions', 'train_one_seed_sharpe')
miss['sequence.py'] = has('backend/models/sequence.py', 'core[:, -1]')      # last-step skip
miss['pretrain.py'] = has('scripts/pretrain.py',
    'from backend.agents.improved_model import', 'freshest available cache')
miss['pipeline.py'] = has('backend/features/pipeline.py', 'build_hybrid_matrix')
miss['engine.py'] = has('backend/backtest/engine.py', 'ratchet state')      # stop-order fix
bad = {k: v for k, v in miss.items() if v}
print('file check:', 'ALL CURRENT' if not bad else f'STALE -> {bad}')
assert not bad, 'Re-upload the flagged file(s) to Drive, then re-run this cell.'

### 2) Dependencies + GPU

In [ ]:
!pip install -q structlog python-dotenv pydantic pydantic-settings requests pyarrow
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (will be slow)')

### 3) Prefetch raw OHLCV into the Drive cache (once)
Downloads Binance monthly CSVs (5m/1h/4h) for the 6 training symbols into
`training_data/raw/*.parquet`. Instant on re-runs — the loader also accepts caches whose
end-month key has rolled forward.

In [ ]:
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "ADAUSDT", "DOGEUSDT"]
START_YEAR = 2022

import pretrain as pre
for s in SYMBOLS:
    print(f'--- {s}')
    dfs = pre.load_full_history(s, START_YEAR, 1, skip_download=False)
    print({tf: len(df) for tf, df in dfs.items()})

### 4) Build + cache the 58-feature hybrid datasets (once)
Separate cell so a runtime disconnect during training never repeats this step — each symbol's
matrix + targets land in `training_data/features/hybrid_*.npz` on Drive.

In [ ]:
import time
import train_quantile_tcn as tq
for s in SYMBOLS:
    t0 = time.time()
    d = tq.build_symbol_dataset(s, START_YEAR, 1)
    print(f'{s}: X{d["X"].shape}  targets{d["y"].shape}  ({time.time()-t0:.0f}s)')

### 5) RUN 1 — pinball objective (quantile edge + uncertainty), 3-seed ensemble

In [ ]:
!python scripts/train_quantile_tcn.py \
    --symbols BTCUSDT ETHUSDT SOLUSDT XRPUSDT ADAUSDT DOGEUSDT \
    --start-year 2022 --epochs 10 --stride 6 --seeds 0,1,2 \
    --hidden 64 --batch 512 --objective pinball

### 6) RUN 2 — DMN sharpe objective (position head, net-Sharpe loss), 3-seed ensemble

In [ ]:
!python scripts/train_quantile_tcn.py \
    --symbols BTCUSDT ETHUSDT SOLUSDT XRPUSDT ADAUSDT DOGEUSDT \
    --start-year 2022 --epochs 12 --seeds 0,1,2 \
    --hidden 64 --objective sharpe

### 7) Side-by-side verdict (re-runs both A/Bs from the saved checkpoints — cheap)

In [ ]:
print('================ PINBALL ================')
!python scripts/train_quantile_tcn.py --ab-only --objective pinball \
    --symbols BTCUSDT ETHUSDT SOLUSDT XRPUSDT ADAUSDT DOGEUSDT --start-year 2022
print('================ SHARPE (DMN) ================')
!python scripts/train_quantile_tcn.py --ab-only --objective sharpe \
    --symbols BTCUSDT ETHUSDT SOLUSDT XRPUSDT ADAUSDT DOGEUSDT --start-year 2022

import glob, torch
print('\nSaved checkpoints (already on Drive — nothing to copy):')
for p in sorted(glob.glob('models/quantile_tcn_overlay_*seed*.pt')):
    ck = torch.load(p, map_location='cpu', weights_only=False)
    print(f"  {p}  objective={ck.get('objective','pinball')}  val_score={ck.get('val_edge_score'):+.3f}")

### 8) How to read the result / what to do next

Each A/B prints `baseline` vs `with QuantileTCN gate` (Sharpe / CAGR / maxDD on the untouched
TEST tail) and a one-line VERDICT.

- **LIFT ≥ +0.05 Sharpe** on either objective → the checkpoints are already in `models/` on
  Drive; sync that folder back to the local repo and we wire the winning ensemble into the live
  TS sleeve behind a flag (paper-shadow first, as always).
- **No lift** → park it. That is a clean, expected outcome (the meta-labeling experiment ended
  the same way) — the 2-sleeve book + risk stack stays the deliverable, and the profit lever
  remains the vol-target / financing work, which needs no model at all.
- Record the run in the trial registry mentally: 2 objectives × 3 seeds — the DSR trial count
  for any promoted result must include both arms.

### Optional — policy-net TCN pretrain (the older A/B, now with an explicit flag)
Only if you also want to refresh the legacy 90-feature policy net; this is the long run
(hours). The unified trainer can now build the TCN trunk directly:
```
!python scripts/pretrain.py --start-year 2022 --epochs 25 --mmap --amp --trunk tcn --skip-download
```